In [1]:
# ============================================================================
# COLAB SETUP â€” run this cell first
# ============================================================================
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # =========================================================================
    DRIVE_PROJECT_PATH = '/content/drive/MyDrive/ENERGIZE'  # <-- EDIT THIS
    # =========================================================================

    import os
    from pathlib import Path
    project_root = Path(DRIVE_PROJECT_PATH)
    if not project_root.exists():
        raise FileNotFoundError(f'Project folder not found: {project_root}')
    os.chdir(project_root)
    sys.path.insert(0, str(project_root))
    print(f'Project root: {project_root}')
else:
    import os
    from pathlib import Path
    project_root = Path(os.getcwd()).parent
    sys.path.insert(0, str(project_root))
    print(f'Running locally. Project root: {project_root}')

Running locally. Project root: c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE


## 1. Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
from pathlib import Path

## 2. Interactive Explorer & Postprocessing Tuner

Load any single prediction CSV (baseline, pruned, fine-tuned, or TFLite INT8)
and explore results **without re-running inference**.

| Widget | Purpose |
|---|---|
| Start / Length | Navigate any window of the test set |
| Threshold | ON/OFF power boundary (W) |
| Min ON / Min OFF | Duration-based denoising knobs |
| Max Length | Optional maximum ON-event length filter |

Metrics are recomputed on the **full** test set every time a slider changes,
so the numbers always reflect the global effect of the chosen parameters.

In [3]:
# ============================================================================
# CONFIGURATION — edit this cell to select which prediction to explore
# ============================================================================
_EXP_MODEL     = 'tcn'        # 'cnn' | 'gru' | 'tcn'
_EXP_APPLIANCE = 'washing_machine'       # 'boiler' | 'ac_1' | 'washing_machine' | 'fridge'
_EXP_STAGE     = 'baseline'   # 'baseline'

# ── Dataset sampling period ────────────────────────────────────────────────
_EXP_PERIOD_SEC = 10          # seconds per sample (PLEGMA = 10 s)

# ── Pull default params from config.py ────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, str(project_root))

from src_pytorch.config import PLEGMA_PARAMS
_ap = PLEGMA_PARAMS[_EXP_APPLIANCE]

_DEF_THRESHOLD  = _ap['threshold']
_DEF_MIN_ON     = _ap.get('min_on',  1)
_DEF_MIN_OFF    = _ap.get('min_off', 1)
_DEF_MAX_LENGTH = _ap.get('min_committed_duration', None)

# ── Resolve CSV path ──────────────────────────────────────────────────────
_exp_prefix = f'{_EXP_MODEL}_{_EXP_APPLIANCE}'
_STAGE_FILES = {
    'baseline': f'{_EXP_APPLIANCE}_predictions.csv',
}
_exp_csv = project_root / 'outputs' / _exp_prefix / 'metrics' / _STAGE_FILES[_EXP_STAGE]

print(f'Model      : {_EXP_MODEL.upper()}')
print(f'Appliance  : {_EXP_APPLIANCE}')
print(f'Stage      : {_EXP_STAGE}')
print(f'CSV        : {_exp_csv}')
print()
print('Default postprocessing params:')
print(f'  Threshold  = {_DEF_THRESHOLD} W')
print(f'  Min ON     = {_DEF_MIN_ON} samples  ({_DEF_MIN_ON * _EXP_PERIOD_SEC} s)')
print(f'  Min OFF    = {_DEF_MIN_OFF} samples  ({_DEF_MIN_OFF * _EXP_PERIOD_SEC} s)')
if _DEF_MAX_LENGTH:
    print(f'  Max Length = {_DEF_MAX_LENGTH} samples  ({_DEF_MAX_LENGTH * _EXP_PERIOD_SEC} s)')
else:
    print('  Max Length : disabled')

Model      : TCN
Appliance  : washing_machine
Stage      : baseline
CSV        : c:\Users\s.athanasoulias\OneDrive - Accenture\Desktop\ENERGIZE\outputs\tcn_washing_machine\metrics\washing_machine_predictions.csv

Default postprocessing params:
  Threshold  = 15 W
  Min ON     = 2 samples  (20 s)
  Min OFF    = 100 samples  (1000 s)
  Max Length = 250 samples  (2500 s)


In [4]:
import numpy as np
import pandas as pd

if not _exp_csv.exists():
    raise FileNotFoundError(
        f'Prediction CSV not found:\n  {_exp_csv}\n'
        'Run the corresponding pipeline step first.'
    )

_exp_df   = pd.read_csv(_exp_csv)
_exp_gt   = _exp_df['ground_truth_W'].values.astype('float32')
_exp_pred = _exp_df['prediction_W'].values.astype('float32')
_exp_N    = len(_exp_gt)

print(f'Loaded {_exp_N:,} samples  '
      f'(~{_exp_N * _EXP_PERIOD_SEC / 3600:.1f} h of data)')
print(f'GT   range : [{_exp_gt.min():.1f}, {_exp_gt.max():.1f}] W')
print(f'Pred range : [{_exp_pred.min():.1f}, {_exp_pred.max():.1f}] W')

Loaded 1,626,600 samples  (~4518.3 h of data)
GT   range : [0.0, 2285.8] W
Pred range : [0.0, 2333.3] W


In [5]:
# ============================================================================
# Interactive Explorer + Postprocessing Tuner
# ============================================================================

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    _HAS_PLOTLY = True
except ImportError:
    _HAS_PLOTLY = False
    print('plotly not found — install with:  pip install plotly')

try:
    import ipywidgets as widgets
    from IPython.display import display as _ipy_display
    _HAS_WIDGETS = True
except ImportError:
    _HAS_WIDGETS = False
    print('ipywidgets not found — install with:  pip install ipywidgets')

from src_pytorch.evaluator import compute_status, compute_metrics

if not (_HAS_PLOTLY and _HAS_WIDGETS):
    raise ImportError('Both plotly and ipywidgets are required for this cell.')


# ── Helper ────────────────────────────────────────────────────────────────
def _fmt_dur(n):
    s = n * _EXP_PERIOD_SEC
    h, r = divmod(int(s), 3600)
    m, s = divmod(r, 60)
    return f'{h:02d}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'


# ── Widgets ───────────────────────────────────────────────────────────────
_CUTOFF = int(_ap['cutoff'])
_wl = widgets.Layout(width='100%')
_ws = {'description_width': '95px'}

_w_start = widgets.IntSlider(
    value=0, min=0, max=max(0, _exp_N - 500), step=500,
    description='Start', continuous_update=False, layout=_wl, style=_ws,
)
_w_len = widgets.IntSlider(
    value=5_000, min=500, max=min(50_000, _exp_N), step=500,
    description='Length', continuous_update=False, layout=_wl, style=_ws,
)
_w_thresh = widgets.IntSlider(
    value=_DEF_THRESHOLD, min=1, max=_CUTOFF, step=1,
    description='Threshold (W)', continuous_update=False, layout=_wl, style=_ws,
)
_w_min_on = widgets.IntSlider(
    value=_DEF_MIN_ON, min=1, max=500, step=1,
    description='Min ON', continuous_update=False, layout=_wl, style=_ws,
)
_w_min_off = widgets.IntSlider(
    value=_DEF_MIN_OFF, min=1, max=500, step=1,
    description='Min OFF', continuous_update=False, layout=_wl, style=_ws,
)
_w_use_maxlen = widgets.Checkbox(
    value=_DEF_MAX_LENGTH is not None,
    description='Use Max Length', indent=False,
    layout=widgets.Layout(width='auto'),
)
_w_max_len = widgets.IntSlider(
    value=_DEF_MAX_LENGTH or 100, min=1, max=2000, step=5,
    description='Max Length', continuous_update=False,
    layout=widgets.Layout(width='100%'), style=_ws,
    disabled=(_DEF_MAX_LENGTH is None),
)
_w_metrics = widgets.HTML(value='<i>Move any slider to refresh metrics.</i>')


# ── Max-length toggle ─────────────────────────────────────────────────────
def _on_maxlen_toggle(change):
    _w_max_len.disabled = not change['new']
_w_use_maxlen.observe(_on_maxlen_toggle, names='value')


# ── Metrics HTML table ────────────────────────────────────────────────────
def _metrics_table(m, thresh, min_on, min_off, max_len):
    f1c = f'{m["f1_complex"]:.4f}' if 'f1_complex' in m else '\u2014'
    rows = [
        ('Threshold',   f'{thresh} W'),
        ('Min ON',      f'{min_on} samples ({_fmt_dur(min_on)})'),
        ('Min OFF',     f'{min_off} samples ({_fmt_dur(min_off)})'),
        ('Max Length',  f'{max_len} samples ({_fmt_dur(max_len)})' if max_len else 'disabled'),
        ('', ''),
        ('MAE',          f'{m["mae"]:.2f} W'),
        ('F1',           f'{m["f1"]:.4f}'),
        ('F1 Complex',   f1c),
        ('Accuracy',     f'{m["accuracy"]:.4f}'),
        ('Precision',    f'{m["precision"]:.4f}'),
        ('Recall',       f'{m["recall"]:.4f}'),
        ('Energy Error', f'{m["energy_error_percent"]:.2f}%'),
    ]
    html = ('<table style="font-size:12px;border-collapse:collapse;'
            'width:100%;margin-top:4px">')
    for k, v in rows:
        if k == '':
            html += '<tr><td colspan="2"><hr style="margin:4px 0"/></td></tr>'
        else:
            html += (
                f'<tr><td style="padding:2px 6px;color:#555;'
                f'white-space:nowrap;font-weight:bold">{k}</td>'
                f'<td style="padding:2px 6px">{v}</td></tr>'
            )
    html += '</table>'
    return html


# ── Build FigureWidget once — updated in-place on every slider change ─────
_x0 = list(range(_w_len.value))
_z0 = [0.0] * _w_len.value

_base = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=['Power (W)', 'ON / OFF Status'],
    vertical_spacing=0.28,
    row_heights=[0.65, 0.35],
)
# Trace 0 – Ground Truth power
_base.add_trace(go.Scatter(
    x=_x0, y=_z0, name='Ground Truth',
    line=dict(color='steelblue', width=1),
    hovertemplate='GT: %{y:.1f} W<extra></extra>',
), row=1, col=1)
# Trace 1 – Prediction power
_base.add_trace(go.Scatter(
    x=_x0, y=_z0, name='Prediction',
    line=dict(color='orangered', width=1),
    hovertemplate='Pred: %{y:.1f} W<extra></extra>',
), row=1, col=1)
# Trace 2 – Threshold line (2-point horizontal segment, no legend entry)
_base.add_trace(go.Scatter(
    x=[0, 1], y=[0, 0], name='Threshold',
    line=dict(color='gray', dash='dash', width=1),
    hoverinfo='skip', showlegend=False,
), row=1, col=1)
# Trace 3 – Pred Status
_base.add_trace(go.Scatter(
    x=_x0, y=_z0, name='Pred Status',
    fill='tozeroy', fillcolor='rgba(255,80,0,0.20)',
    line=dict(color='orangered', width=1.2),
    hovertemplate='Pred: %{y}<extra></extra>',
), row=2, col=1)
# Trace 4 – GT Status
_base.add_trace(go.Scatter(
    x=_x0, y=_z0, name='GT Status',
    fill='tozeroy', fillcolor='rgba(70,130,180,0.25)',
    line=dict(color='steelblue', width=2.5),
    hovertemplate='GT: %{y}<extra></extra>',
), row=2, col=1)

_base.update_layout(
    height=620,
    margin=dict(t=120, b=40, l=60, r=20),
    title=dict(text='Loading\u2026', y=0.98, yanchor='top'),
    legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='right', x=1),
    hovermode='x unified',
)
_base.update_yaxes(title_text='Power (W)', row=1, col=1)
_base.update_yaxes(
    title_text='Status', tickvals=[0, 1], ticktext=['OFF', 'ON'],
    range=[-0.15, 1.15], row=2, col=1,
)
_base.update_xaxes(title_text='Sample index', row=2, col=1)

_fig_widget = go.FigureWidget(_base)


# ── Refresh callback — mutates FigureWidget in-place ─────────────────────
def _refresh(_change=None):
    start   = _w_start.value
    length  = _w_len.value
    thresh  = _w_thresh.value
    min_on  = _w_min_on.value
    min_off = _w_min_off.value
    max_len = _w_max_len.value if _w_use_maxlen.value else None
    end     = min(start + length, _exp_N)

    gt_st   = compute_status(_exp_gt,   thresh, min_on, min_off, max_len)
    pred_st = compute_status(_exp_pred, thresh, min_on, min_off, max_len)
    metrics = compute_metrics(
        _exp_gt, _exp_pred, thresh,
        min_on=min_on, min_off=min_off, min_committed_duration=max_len,
    )

    _w_metrics.value = _metrics_table(metrics, thresh, min_on, min_off, max_len)

    x = list(range(start, end))

    with _fig_widget.batch_update():
        _fig_widget.data[0].x = x
        _fig_widget.data[0].y = _exp_gt[start:end].tolist()
        _fig_widget.data[1].x = x
        _fig_widget.data[1].y = _exp_pred[start:end].tolist()
        _fig_widget.data[2].x = [x[0], x[-1]]
        _fig_widget.data[2].y = [thresh, thresh]
        _fig_widget.data[3].x = x
        _fig_widget.data[3].y = pred_st[start:end].tolist()
        _fig_widget.data[4].x = x
        _fig_widget.data[4].y = gt_st[start:end].tolist()
        _fig_widget.layout.title.text = (
            f'<b>{_EXP_APPLIANCE.replace("_"," ").title()} \u2014 '
            f'{_EXP_MODEL.upper()} \u2014 {_EXP_STAGE.title()}</b>'
            f'  |  samples {start:,}\u2013{end:,} / {_exp_N:,}'
        )


for _w in [_w_start, _w_len, _w_thresh, _w_min_on, _w_min_off, _w_max_len]:
    _w.observe(_refresh, names='value')
_w_use_maxlen.observe(_refresh, names='value')


# ── Layout ────────────────────────────────────────────────────────────────
_sidebar = widgets.VBox(
    [
        widgets.HTML('<b style="font-size:13px">Navigation</b>'),
        _w_start, _w_len,
        widgets.HTML(
            '<hr style="margin:8px 0"/>'
            '<b style="font-size:13px">Postprocessing params</b>'
        ),
        _w_thresh, _w_min_on, _w_min_off,
        widgets.HBox([_w_use_maxlen, _w_max_len]),
        widgets.HTML(
            '<hr style="margin:8px 0"/>'
            '<b style="font-size:13px">Metrics (full test set)</b>'
        ),
        _w_metrics,
    ],
    layout=widgets.Layout(
        width='320px', min_width='320px',
        padding='10px', margin='0 10px 0 0',
        border='1px solid #ddd',
        overflow_y='auto',
    ),
)
_main = widgets.Box(
    [_fig_widget],
    layout=widgets.Layout(flex='1 1 auto', min_width='0'),
)
_ipy_display(widgets.HBox([_sidebar, _main]))
_refresh()